In [73]:
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer



model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B")



In [74]:
import torch

def generate(
        model, tokenizer, input_ids, max_length=50, do_sample=False,top_k = None
):
    """Generate a sequence without using model.generate()
    
    Args:
        model: The model used for generation
        tokenizer: tokenizer for the model
        input_ids: The input IDsm  
        max_length: max_length of generated sequence
        do_sample: whether to use sampling
        tok_k: The number of tokens to sample from
    """
    generation = input_ids
    if not do_sample:
        #Greedy generation
        for i in range(max_length):
            output = model(generation)
            next_token = output.logits[:, -1, :].argmax(dim=1).unsqueeze(0)
            generation = torch.cat((generation, next_token), dim=1)
    elif do_sample:
        #Sampling
        for i in range(max_length):
            if top_k != None:
                #Top_k sampling
                output = model(generation)
                next_token_sample_space = torch.topk(output.logits[:, -1, :],top_k)
                cumdf = next_token_sample_space.values.softmax(dim=1).cumsum(1)
                idx = torch.searchsorted(cumdf.flatten(),torch.rand(1))
                next_token = next_token_sample_space.indices[:,idx]
                generation = torch.cat((generation, next_token), dim=1)
            else:
                #Sampling
                output = model(generation)
                next_token_sample_space = output.logits[:, -1, :]
                cumdf = next_token_sample_space.softmax(dim=1).cumsum(1)
                idx = torch.searchsorted(cumdf.flatten(),torch.rand(1))
                generation = torch.cat((generation, torch.unsqueeze(idx,0)), dim=1)




    return tokenizer.decode(generation[0])
    

In [75]:
torch.manual_seed(42)

#Greedy search
input_ids = tokenizer("Hello, my name is ",return_tensors="pt")["input_ids"]
generation = generate(model,tokenizer,input_ids,max_length=50,do_sample=False)

print(generation)

Hello, my name is 100% me. I am a 100% real person. I am a 100% real person. I am a 100% real person. I am a 100% real person.


In [76]:
torch.manual_seed(42)


#Sampling
input_ids = tokenizer("Hello, my name is ",return_tensors="pt")["input_ids"]
generation = generate(model,tokenizer,input_ids,max_length=50,do_sample=True)

print(generation)

Hello, my name is ashioko and here is my school letter frequency. hopefully you enjoy it!

If your question is too hard to ask then this is the way to be inviting. :) 

--------------------------------------------
Please submit your answer by following this buttons once and click Next button.


In [77]:
torch.manual_seed(42)

#Sampling top_k = 5
input_ids = tokenizer("Hello, my name is ",return_tensors="pt")["input_ids"]
generation = generate(model,tokenizer,input_ids,max_length=50,do_sample=True,top_k=5)

print(generation)

Hello, my name is 456897 and I’m a little boy who loves to play. I’m 12 years old and I live in a small city called 1234. I have 3 brothers, they are called 45
